# 04.03 ACT 训练与过程可视化

## 本节概述

<table style="text-align: left; margin-left: 0;">
<tr><td align="left"><b>前置要求</b></td><td align="left">已完成 04.02（已下载数据集，LeRobot 已就绪）</td></tr>
<tr><td align="left"><b>本节目标</b></td><td align="left">在昇腾 NPU 上训练 ACT 策略，解读训练曲线</td></tr>
<tr><td align="left"><b>本节内容</b></td><td align="left">NPU 训练环境准备 → 启动训练（smoke + 完整）→ 训练曲线可视化</td></tr>
</table>

> ⏱️ 完整训练（5000 步）在昇腾 NPU 上约需 2-4 小时。本节提供 smoke 测试（20 步，约 2-3 分钟）用于验证流程。

## 🔧 昇腾 NPU 训练（已验证 ✅）

> LeRobot 官方**不支持昇腾 NPU**（`is_torch_device_available` 硬编码只认 cuda/mps/xpu/cpu）。
> 本章提供了基于 **CANN 官方样例**裁剪的 NPU 补丁脚本，**已在 CANNLab 8.5.2 + Ascend910B3 实测通过**。

**NPU 训练必须在终端操作**（不是 notebook），因为需要 clone lerobot 源码并打补丁。请在终端按以下 3 步操作：

```bash
cd contrib/tutorials/ascend_multimodal_practice/04_vla_lerobot

# 【第 1 步】准备 NPU 训练环境（首次约 5-10 分钟）
# 下载 lerobot 源码（从 GitCode）+ 应用 NPU 补丁 + 装依赖
bash src/npu_support/scripts/setup_lerobot_npu.sh

# 【第 2 步】缓存 ResNet18 权重（重要！否则训练会卡在下载）
mkdir -p src/npu_support/.cache/torch/hub/checkpoints/
curl -L -o src/npu_support/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth \
  "https://download.pytorch.org/models/resnet18-f37072fd.pth"

# 【第 3 步】smoke 测试（20步，约 2-3 分钟，验证 NPU 训练链路）
bash src/npu_support/scripts/run_train_npu.sh act_so101_smoke

# 【第 4 步】查看训练日志（确认训练进度和结果）
tail -f src/npu_support/logs/train_act_so101_smoke_*.log
```

**怎么看训练是否完成？**

训练用后台进程启动（`nohup`），启动后会立刻返回终端。用上面的 `tail -f` 实时查看日志，出现以下内容说明训练成功完成：

```text
Training: 100%|██████████| 20/20 [01:12<00:00, 3.67s/it, grad_norm=217.59, loss=10.2368]
INFO ... Checkpoint policy after step 20
INFO ... End of training        ← 看到这行就完成了，按 Ctrl+C 退出 tail
```

> 💡 loss 从约 89 降到约 10 属于正常（smoke 只跑 20 步，不追求效果，验证链路通即可）。
> 💡 训练产物保存在 `src/ckpt/act_so101_smoke_<时间戳>/`，完成后可运行下方 Cell[1] 检查。

> ✅ **实测结果**（CANNLab Ascend910B3）：smoke 20 步约 73 秒完成，loss 从 89.5 → 10.2（降 88%），训练链路完全打通。

> 💡 完整说明（含 8 个常见问题 FAQ）见 [`src/npu_support/SETUP_NPU.md`](./src/npu_support/SETUP_NPU.md)。
> 💡 NPU 训练已针对 CANNLab 做了关键优化：用 pyav 替代 torchcodec、num_workers=0、从 GitCode 下载源码。

---

## 1. 训练配置说明

ACT 训练用 `lerobot-train` 命令（通过 NPU 补丁脚本启动），核心配置：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <th align="left">配置项</th><th align="left">含义</th><th align="left">本节取值</th></tr>
<tr><td align="left"><code>dataset.repo_id</code></td><td align="left">数据集标识</td><td align="left"><code>local/so101_block</code></td></tr>
<tr><td align="left"><code>dataset.root</code></td><td align="left">数据集本地路径</td><td align="left"><code>./src/data_final</code></td></tr>
<tr><td align="left"><code>policy.type</code></td><td align="left">策略类型</td><td align="left"><code>act</code>（本节用 ACT）</td></tr>
<tr><td align="left"><code>policy.device</code></td><td align="left">训练设备</td><td align="left"><code>npu</code>（昇腾，通过补丁支持）</td></tr>
<tr><td align="left"><code>dataset.video_backend</code></td><td align="left">视频解码</td><td align="left"><code>pyav</code>（NPU 兼容，非 torchcodec）</td></tr>
<tr><td align="left"><code>num_workers</code></td><td align="left">数据加载进程</td><td align="left"><code>0</code>（避免 CANNLab 共享内存不足）</td></tr>
<tr><td align="left"><code>steps</code></td><td align="left">训练步数</td><td align="left">5000（完整）/ 20（smoke）</td></tr>
<tr><td align="left"><code>batch_size</code></td><td align="left">批大小</td><td align="left">8（显存不足可调小）</td></tr>
</table>

配置文件在 `src/npu_support/configs/` 下（smoke 和完整各一个），上述优化已固化在配置中。



In [ ]:
# ===== 检查 smoke 训练结果 =====
# 从训练日志确认 smoke 是否成功完成（比找checkpoint更可靠）
import re, glob

smoke_logs = sorted(glob.glob("src/npu_support/logs/train_act_so101_smoke_*.log"))
if smoke_logs:
    latest_log = smoke_logs[-1]
    print(f"✅ 找到 smoke 训练日志: {latest_log}")
    
    # 解析训练结果
    with open(latest_log) as f:
        content = f.read()
    
    # 检查是否训练完成
    if "End of training" in content:
        print("✅ smoke 训练成功完成！")
    else:
        print("⚠️ 日志中未发现 'End of training'，训练可能未完成或中断")
    
    # 提取 loss 趋势
    losses = []
    for m in re.finditer(r'step:(\d+).*?loss:([\d.]+)', content):
        step = int(m.group(1))
        loss = float(m.group(2))
        if step not in [s for s, _ in losses]:  # 去重
            losses.append((step, loss))
    
    if losses:
        print(f"\n训练 loss 趋势（共 {len(losses)} 步）:")
        print(f"  初始 loss: {losses[0][1]:.3f} (step {losses[0][0]})")
        print(f"  最终 loss: {losses[-1][1]:.3f} (step {losses[-1][0]})")
        if losses[0][1] > 0:
            drop = (losses[0][1] - losses[-1][1]) / losses[0][1] * 100
            print(f"  下降幅度: {drop:.1f}%")
        print(f"\n💡 loss 下降说明 NPU 训练链路完全打通！")
        print(f"💡 接下来可运行下方 Cell[5] 查看 loss 曲线，或跑完整训练。")
else:
    print("⚠️ 未找到 smoke 训练日志（src/npu_support/logs/train_act_so101_smoke_*.log）")
    print()
    print("请先在终端按 4 步完成 smoke 训练（详见上方说明）:")
    print("  第1步: bash src/npu_support/scripts/setup_lerobot_npu.sh")
    print("  第2步: curl 下载 ResNet18 权重（见 SETUP_NPU.md 第2步）")
    print("  第3步: bash src/npu_support/scripts/run_train_npu.sh act_so101_smoke")
    print("  第4步: tail -f src/npu_support/logs/train_act_so101_smoke_*.log （看进度）")

## 2. 完整训练（5000 步，约 2-4 小时）

smoke 通过后，可启动完整训练（后台跑，关掉终端不影响）：

```bash
bash src/npu_support/scripts/run_train_npu.sh act_so101
```

> ⏱️ 完整训练耗时 2-4 小时，脚本用 `nohup` 后台启动。
> 💡 实时看进度：`tail -f src/npu_support/logs/train_act_so101_*.log`
> 💡 训练产物在 `src/ckpt/act_so101_<时间戳>/` 下（含 checkpoints / final / config.json）
> 💡 如果不想自己训练，可直接使用课程提供的预训练模型（见 04.04 节）。

In [ ]:
# ===== 检查完整训练结果 =====
# 完整训练（5000步）完成后，本 cell 帮你确认产物
import os, glob

# 找最新的完整训练 checkpoint（排除 smoke）
full_ckpts = sorted([d for d in glob.glob("src/npu_support/ckpt/act_so101_*") if "smoke" not in d])
if full_ckpts:
    latest = full_ckpts[-1]
    print(f"✅ 找到完整训练产物: {latest}")
    for root, dirs, files in os.walk(latest):
        for f in files[:10]:  # 只显示前10个文件
            fpath = os.path.join(root, f)
            size = os.path.getsize(fpath) / 1024**2
            rel = os.path.relpath(fpath, latest)
            print(f"  {rel} ({size:.1f} MB)")
    print()
    print("💡 完整训练完成！可到 04.04 做离线推理评测。")
else:
    print("⚠️ 未找到完整训练产物（src/npu_support/ckpt/act_so101_*）")
    print("💡 完整训练需 2-4 小时。如不想等待，可直接用预训练模型（见 04.04）。")
    print()
    print("启动完整训练（终端运行）：")
    print("  bash src/npu_support/scripts/run_train_npu.sh act_so101")

## 3. 训练过程可视化

训练过程中，LeRobot 会输出日志。我们解析训练目录下的日志文件，画出 loss 曲线：

In [ ]:
# ===== 解析训练日志，绘制 loss 曲线 =====
# 从 NPU 训练日志（src/npu_support/logs/）解析 loss 数据画曲线
# 同时支持 smoke（20步）和完整训练（5000步）的日志
import re, os, glob
import pandas as pd
import matplotlib.pyplot as plt

def parse_loss_from_log(log_path):
    """从训练日志解析 step 和 loss"""
    steps, losses, grad_norms = [], [], []
    seen_steps = set()  # 去重（日志可能有重复行）
    with open(log_path) as f:
        for line in f:
            # 匹配 "step:N ... loss:XX.XXX ... grdn:G.GGG"
            m = re.search(r'step:(\d+).*?loss:([\d.]+).*?grdn:([\d.]+)', line)
            if m:
                step = int(m.group(1))
                if step not in seen_steps:
                    seen_steps.add(step)
                    steps.append(step)
                    losses.append(float(m.group(2)))
                    grad_norms.append(float(m.group(3)))
    return steps, losses, grad_norms

# 找训练日志（smoke 和完整）
smoke_logs = sorted(glob.glob("src/npu_support/logs/train_act_so101_smoke_*.log"))
full_logs = sorted(glob.glob("src/npu_support/logs/train_act_so101_*.log"))
# 排除 smoke 的才是完整训练
full_logs = [l for l in full_logs if "smoke" not in l]

# 优先显示完整训练，其次 smoke
all_logs = full_logs + smoke_logs

if all_logs:
    # 用最新的日志
    latest_log = all_logs[0] if full_logs else all_logs[-1]
    train_type = "Full Training" if "smoke" not in latest_log else "Smoke Test"
    print(f"找到训练日志: {latest_log} ({train_type})")
    
    steps, losses, grad_norms = parse_loss_from_log(latest_log)
    
    if losses:
        print(f"解析到 {len(losses)} 个 step 的 loss 数据")
        print(f"  初始 loss: {losses[0]:.3f}")
        print(f"  最终 loss: {losses[-1]:.3f}")
        if losses[0] > 0:
            drop = (losses[0] - losses[-1]) / losses[0] * 100
            print(f"  下降幅度: {drop:.1f}%")
        
        # 画双图：loss + grad_norm
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # loss 曲线
        axes[0].plot(steps, losses, color='steelblue', alpha=0.7, linewidth=1, label='loss')
        if len(losses) > 20:
            # 数据多时加移动平均
            window = max(10, len(losses)//10)
            ma = pd.Series(losses).rolling(window).mean()
            axes[0].plot(steps, ma, color='red', linewidth=2, label=f'{window}-step moving avg')
        axes[0].set_xlabel('Training Step')
        axes[0].set_ylabel('Loss')
        axes[0].set_title(f'ACT Training Loss ({train_type}, {len(losses)} steps)')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # grad_norm 曲线
        axes[1].plot(steps, grad_norms, color='orange', alpha=0.7, linewidth=1, label='grad_norm')
        axes[1].set_xlabel('Training Step')
        axes[1].set_ylabel('Gradient Norm')
        axes[1].set_title(f'Gradient Norm ({train_type})')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print(f"\n💡 loss 持续下降说明训练正常。完整训练（5000步）loss 会降到更低。")
        if train_type == "smoke测试":
            print(f"💡 smoke 只是验证链路（20步），想看更好的效果请跑完整训练。")
    else:
        print("⚠️ 日志中未找到 loss 记录")
        print("💡 可能训练还在进行中，请等训练完成后再运行本 cell。")
else:
    print("=" * 60)
    print("⚠️ 未找到训练日志")
    print("=" * 60)
    print("已查找路径: src/npu_support/logs/train_act_so101_*.log")
    print()
    print("💡 请先在终端完成训练：")
    print("   bash src/npu_support/scripts/run_train_npu.sh act_so101_smoke   # smoke(20步)")
    print("   bash src/npu_support/scripts/run_train_npu.sh act_so101          # 完整(5000步)")
    print()
    print("下面展示一个示意曲线（理解 loss 下降趋势）:")
    import numpy as np
    np.random.seed(42)
    demo_steps = range(1, 501)
    demo_loss = 2.0 * np.exp(-np.linspace(0, 5, 500)) + 0.1 + np.random.randn(500)*0.05
    plt.figure(figsize=(10, 5))
    plt.plot(list(demo_steps), demo_loss, color='steelblue', alpha=0.5, linewidth=1, label='loss (demo)')
    pd.Series(demo_loss).rolling(30).mean().plot(color='red', linewidth=2, label='moving avg (demo)')
    plt.xlabel('Training Step'); plt.ylabel('Loss')
    plt.title('ACT Training Loss Curve (Demo - run training to see real curve)')
    plt.legend(); plt.grid(True, alpha=0.3); plt.show()
    print("💡 完成训练后重新运行本 cell，会显示真实 loss 曲线。")

### 训练曲线解读

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <th align="left">观察点</th><th align="left">正常表现</th><th align="left">异常处理</th></tr>
<tr><td align="left">Loss 趋势</td><td align="left">持续下降并趋于平稳</td><td align="left">若不降：检查数据加载、学习率</td></tr>
<tr><td align="left">Loss 波动</td><td align="left">小幅波动正常</td><td align="left">剧烈震荡：减小学习率或增大 batch_size</td></tr>
<tr><td align="left">最终 Loss</td><td align="left">ACT 通常降到 0.1-0.5</td><td align="left">若过高：增加训练步数或数据量</td></tr>
</table>

## 4. 训练产物说明

训练完成后，`outputs/train/act_so101/` 目录包含：

<table style="text-align: left; margin-left: 0;">
<tr><th align="left">文件</th><th align="left">说明</th></tr>
<tr><td align="left"><code>checkpoints/</code></td><td align="left">训练过程中的模型快照（每 save_freq 步保存）</td></tr>
<tr><td align="left"><code>final/</code></td><td align="left">训练结束时的最终模型</td></tr>
<tr><td align="left"><code>stats.json</code></td><td align="left">训练统计（loss、eval 结果）</td></tr>
<tr><td align="left"><code>config.json</code></td><td align="left">训练配置（用于复现）</td></tr>
</table>

下一节我们将加载训练好的模型做离线测试。

---

## 本节练习

**练习 1（选择）**：NPU 训练时 `policy.device` 应该设为什么？
- A. 指定训练用 CPU
- B. 指定为 npu（昇腾，需通过补丁支持）
- C. 指定数据集路径
- D. 指定保存路径

**练习 2（填空）**：ACT 完整训练建议 ______ 步起，在昇腾 NPU 上约需 ______ 小时；显存不足时应减小 ______ 参数。

**练习 3（简答）**：如果训练 loss 一直不下降，可能的原因有哪些？至少列出 2 个。

> 💡 参考答案见下方 code cell。

In [ ]:
# 查看本节练习答案
!cat ./answer/04.03_training/answers.txt
